# RaCA on Kaggle: train and evaluate one configuration

**Before running**
1. *Add Input* → attach the **scenes** dataset (output of the step-2 subset notebook; it contains `kitti_subset/selection.json`).
2. *Add Input* → attach the **before-run** dataset (`mra-before-run`: `ckpts/warm_start.pth` and `runs/before/results_trained.json`) so this run starts from the same weights and is compared with original MRA automatically (without it the warm start is rebuilt from SGN-T).
3. *Settings*: Accelerator **GPU T4 x2** (or T4 x1), Internet **On**. `train.sh` uses every GPU it sees.
4. Set `RUN` below (`mra_before`, `raca_c2c3c4`, `raca_full`, `raca_c1`, `raca_c1_fine`, `raca_c2`, `raca_c2c3`, `raca_c4`) and click *Run All*. When `Epoch [1][50/400]` lines appear, stop the session and use *Save Version → Save & Run All (Commit)* so the ~5 h run continues in the background.

All runs start from the same SGN-T warm start with seed 0. MRA's own initialisation does not fit a T4: a random occupancy head seeds about half the voxel grid and the first step needs > 13 GB (see `kaggle/README.md`). `--deterministic` is off by default (`DETERMINISTIC=1` in `train.sh`).

Everything this notebook does is in the repository's `kaggle/` folder; see `kaggle/README.md`.

In [ ]:
# ---- Settings ----
RUN      = "raca_full"     # which kaggle/configs/<RUN>.py to train
REPO_URL = "https://github.com/fafnirLore/RaCA.git"
BRANCH   = "farquleet_add_module"
EXTRA    = ""                 # extra train.py args, e.g. "--cfg-options runner.max_epochs=3"
import os
REPO = f"/kaggle/working/RaCA"
os.environ.update(dict(RUN=RUN, REPO=REPO, REPO_URL=REPO_URL, BRANCH=BRANCH, EXTRA=EXTRA))
print("run:", RUN, "| repo:", REPO)

In [ ]:
%%bash
# 1. Code + software (~15 min the first time; skipped when already installed in this session)
nvidia-smi --query-gpu=name,memory.total --format=csv
if [ -d "$REPO/.git" ]; then cd "$REPO" && git fetch -q origin "$BRANCH" && git checkout -q "$BRANCH" && git reset -q --hard "origin/$BRANCH"
else git clone -q -b "$BRANCH" "$REPO_URL" "$REPO" && cd "$REPO"; fi
git log --oneline -1    # must show the commit you expect; an existing clone is reset to the branch head on GitHub
bash kaggle/install.sh && bash kaggle/get_weights.sh sgn

In [ ]:
# 2. Data: find the saved subset and rebuild MRA's file layout at /tmp/kitti_full/ (~3 min)
import glob, json, subprocess, os
hits = glob.glob("/kaggle/input/**/kitti_subset/selection.json", recursive=True)
assert hits, "scenes dataset not attached (see the top of this notebook)"
SRC = os.path.dirname(hits[0]) + "/"
s = json.load(open(hits[0]))["summary"]
print("subset:", SRC, f"| train frames {s['train']['frames']} | val frames {s['val']['frames']}")
if not os.path.exists("/tmp/kitti_full/dataset/labels"):
    subprocess.run(["python", f"{REPO}/kaggle/expand_subset.py", SRC, "/tmp/kitti_full/"], check=True)

In [ ]:
# 3. Starting weights: reuse the 'before' run's warm_start.pth when attached, otherwise build it from SGN-T (~1 min).
#    Shared by every run; MRA's own initialisation does not fit a T4 (see the top of this notebook).
import glob, shutil, subprocess, os
os.makedirs(f"{REPO}/ckpts", exist_ok=True)
ws = glob.glob("/kaggle/input/**/ckpts/warm_start.pth", recursive=True)
if ws:
    shutil.copy(ws[0], f"{REPO}/ckpts/warm_start.pth"); print("starting weights copied from", ws[0])
else:
    print("building starting weights from SGN-T")
    subprocess.run(f"cd {REPO} && CUDA_VISIBLE_DEVICES=0 /tmp/mra_env/bin/python kaggle/make_warm_start.py ckpts/warm_start.pth "
                   "2>&1 | grep -v 'mmcv - INFO\|is the same before\|torch.Size\|^ *$\|PretrainedInit\|XavierInit' | tail -4",
                   shell=True, check=True)
BEFORE = glob.glob("/kaggle/input/**/results_trained.json", recursive=True)
os.environ["BEFORE_JSON"] = BEFORE[0] if BEFORE else ""
print("'before' results for comparison:", BEFORE[0] if BEFORE else "none attached")


In [ ]:
%%bash
# 4. Smoke test: 2 real training steps (~2 min). Stops the notebook here if the configuration does not run.
cd "$REPO"
PYTHONPATH=$REPO CUDA_VISIBLE_DEVICES=0 timeout 20m /tmp/mra_env/bin/python kaggle/smoke_test.py kaggle/configs/$RUN.py --steps 2 \
  2>&1 | grep -v -E "mmcv - INFO|is the same before|torch.Size|PretrainedInit|XavierInit|UserWarning|floor_divide|return torch|^ *$"
test ${PIPESTATUS[0]} -eq 0

In [ ]:
%%bash
# 5. Train (~40-55 min per epoch on one T4, about half on T4 x2, + ~5 min validation per epoch). All visible GPUs are
#    used; on 2 GPUs train.sh scales lr x2 and warm-up /2 (GPUS=1 forces one GPU, SCALE_LR=0 keeps the config values). Lines like `Epoch [1][50/400] ... loss: ...`
#    show progress; `EarlyStoppingHook: ...` lines show the validation mIoU and the patience counter.
cd "$REPO" && bash kaggle/train.sh kaggle/configs/$RUN.py $RUN $EXTRA

In [ ]:
%%bash
# 6. Evaluate on the 150 validation frames (~5 min): the best checkpoint kept by the evaluation hook, else the last one
cd "$REPO"
CKPT=$(ls -t /kaggle/working/runs/$RUN/best_*.pth 2>/dev/null | head -1); CKPT=${CKPT:-/kaggle/working/runs/$RUN/latest.pth}
echo "evaluating $CKPT"
bash kaggle/eval.sh kaggle/configs/$RUN.py "$CKPT" /kaggle/working/runs/$RUN/results_trained.json "$RUN: trained"


In [ ]:
%%bash
# 7. Compare with the original MRA run when its results are attached
cd "$REPO"
if [ -n "$BEFORE_JSON" ]; then python kaggle/compare.py "$BEFORE_JSON" /kaggle/working/runs/$RUN/results_trained.json | tee /kaggle/working/runs/$RUN/comparison.txt
else echo "no 'before' results attached - compare by hand with kaggle/compare.py"; fi

In [ ]:
%%bash
# 8. What is kept as the notebook output (checkpoint, logs, results); the dataset copy in /tmp is not saved
du -sh /kaggle/working/runs/$RUN; ls /kaggle/working/runs/$RUN